# Notebook 07 — LSTM vs FinBERT-LSTM (Scoped ke Batasan Masalah Skripsi BAB I)

**Sumber kebenaran scope:** `code_plan/skripsi_scoped_lstm_finbert_plan.md` dan BAB I skripsi
("PREDIKSI PERGERAKAN INDEKS PASAR SAHAM BERDASARKAN SENTIMEN BERITA KEUANGAN DI YAHOO
FINANCE MENGGUNAKAN MODEL FINBERT DAN LONG SHORT-TERM MEMORY").

Notebook ini BERBEDA dari notebook 03/05/06 (metodologi "v5") yang sengaja jauh lebih ketat
secara metodologis (walk-forward, AUC-ROC, ensemble, kalibrasi Platt). Notebook ini
mereplikasi **persis** apa yang diminta Batasan Masalah BAB I — tidak lebih, tidak kurang:

- Horizon prediksi: **1 hari** (label = 1 jika `Close[t+1] > Close[t]`), tanpa filter zona netral.
- Fitur: **OHLCV mentah + skor sentimen FinBERT harian** — TANPA indikator teknikal (RSI/MACD/BB).
- Tuning hyperparameter: **Random Search** (bukan Grid Search), 20 iterasi per model.
- Split: **kronologis persentase 70/15/15** (bukan walk-forward).
- Evaluasi: **HANYA Accuracy, Precision, Recall, F1-Score** (`sklearn.metrics`).
- Dua model dibandingkan berdampingan:
  - **Model A (baseline)** — LSTM murni, fitur OHLCV saja.
  - **Model B (FinBERT-LSTM)** — LSTM + skor sentimen FinBERT sebagai fitur tambahan.

**Keputusan desain yang sudah dikonfirmasi (lihat log sesi `memory/session_decisions.md`):**
1. Split train/val/test: persentase kronologis 70/15/15.
2. "Fungsi aktivasi" pada Random Search diinterpretasikan sebagai aktivasi FC-head
   (`{ReLU, Tanh, Sigmoid}`) — `nn.LSTM` PyTorch tidak native expose recurrent-activation
   yang bisa diganti tanpa custom cell, jadi interpretasi ini yang paling natural & benar
   secara teknis tanpa over-engineering.
3. Kriteria seleksi Random Search: **Val Accuracy** tertinggi.
4. Jumlah iterasi Random Search: **20** per model.
5. Output LSTM sebelum dense head: **last-timestep** (konsisten dengan model produksi
   `direction_classifier.pt`, bukan mean-pooling seperti notebook v5).
6. Window size sliding-window: **fixed 10** (tidak ikut di-*search*, konsisten notebook05;
   BAB I tidak menyebut window size sebagai bagian Random Search).

**Prinsip anti p-hacking (konsisten dengan seluruh notebook sebelumnya di project ini):**
hasil dilaporkan apa adanya. Kalau Model B (FinBERT-LSTM) ternyata TIDAK mengungguli
Model A (LSTM-OHLCV), itu tetap temuan valid yang harus dilaporkan jujur di BAB IV/V — bukan
berarti harus "diakali" supaya Model B menang.

**Status: notebook ini ditulis LENGKAP tapi BELUM DIEKSEKUSI.** Semua cell `outputs=[]`.
Eksekusi disarankan lewat `papermill`/`nbconvert --execute` headless (bukan cell-by-cell
Jupyter MCP interaktif) untuk hemat token, sebagai langkah terpisah setelah notebook ini
direview.


## Bagian 1 — Data

Reuse fungsi yang sudah ada di `pipeline/feature_engineering/dataset_builder.py`
(TIDAK reinvent):
- `load_joined()` → gabungan OHLCV + sentimen harian, kolom `FEATURE_COLS`
  (`Open, High, Low, Close, Volume, sentiment_mean`) — dipakai Model B.
- `load_ohlcv_only()` → OHLCV saja, kolom `OHLCV_FEATURE_COLS`
  (`Open, High, Low, Close, Volume`) — dipakai Model A.

Kedua fungsi ini membaca dari file yang SUDAH diproses `StockPreprocessor.clean()`
(`data/processed/stocks/GSPC_2018_2024.csv`, rentang 2018-01-01 s.d. 2024-03-04) dan TIDAK
menghitung indikator teknikal apa pun — sudah sesuai Batasan Masalah BAB I tanpa perlu
modifikasi.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
# Jika notebook dijalankan dari dalam folder notebooks/, root project adalah parent-nya
if (PROJECT_ROOT / "pipeline").exists() is False:
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

import logging
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from pipeline.feature_engineering.dataset_builder import (
    FEATURE_COLS,
    OHLCV_FEATURE_COLS,
    load_joined,
    load_ohlcv_only,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("notebook07")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Device: %s", DEVICE)


2026-07-23 18:05:21,406 [INFO] notebook07: Device: cuda


In [2]:
def set_seed(seed: int) -> None:
    """Set semua seed relevan untuk reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

df_ohlcv = load_ohlcv_only()   # Model A — OHLCV_FEATURE_COLS
df_joined = load_joined()      # Model B — FEATURE_COLS (OHLCV + sentiment_mean)

print("Model A (OHLCV-only) —", df_ohlcv.shape, "kolom:", list(df_ohlcv.columns))
print("Model B (FinBERT-LSTM) —", df_joined.shape, "kolom:", list(df_joined.columns))
print("Rentang tanggal:", df_joined["Date"].min().date(), "s.d.", df_joined["Date"].max().date())


2026-07-23 18:05:21,437 [INFO] pipeline.feature_engineering.dataset_builder: OHLCV-only dataset: 1552 rows (2018-01-02 – 2024-03-04)
2026-07-23 18:05:21,457 [INFO] pipeline.feature_engineering.dataset_builder: Stock loaded: 1552 rows (2018-01-02 – 2024-03-04)
2026-07-23 18:05:21,459 [INFO] pipeline.feature_engineering.dataset_builder: Sentiment loaded: 3504 rows (2008-01-02 – 2024-03-04)
2026-07-23 18:05:21,465 [INFO] pipeline.feature_engineering.dataset_builder: Joined dataset: 1528 rows (2018-01-02 – 2024-03-04)


Model A (OHLCV-only) — (1552, 6) kolom: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
Model B (FinBERT-LSTM) — (1528, 7) kolom: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'sentiment_mean']
Rentang tanggal: 2018-01-02 s.d. 2024-03-04


## Bagian 1a — Analisis Eksplorasi Data (EDA)

Sebelum label/split/training, pahami dulu karakteristik data mentah: missing values,
gap tanggal, outlier, distribusi, korelasi antar fitur, dan yang **paling penting** —
korelasi fitur terhadap label (arah harga besok). Ini untuk mendiagnosis kenapa Model A/B
di Bagian 4-5 collapse ke prediksi kelas konstan berapa pun konfigurasi Random Search-nya.

Semua plot disimpan ke file `notebooks/eda07_*.png` (bukan inline), sesuai konvensi project
(`user_preferences.md`). Analisis di sini bersifat **read-only** — tidak mengubah variabel
`df_ohlcv`/`df_joined` yang dipakai Bagian 1b (Label & Split) di bawah.


In [3]:
import matplotlib
matplotlib.use("Agg")  # render ke file, bukan inline — konsisten konvensi project
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

EDA_DIR = PROJECT_ROOT / "notebooks"


### Missing Values & Gap Tanggal

In [4]:
print("=== Missing Values — df_ohlcv (Model A) ===")
print(df_ohlcv.isna().sum())

print("\n=== Missing Values — df_joined (Model B) ===")
print(df_joined.isna().sum())

# Gap tanggal: selisih hari antar baris berturutan (indikasi hari trading yang hilang / libur panjang)
date_diff = df_joined["Date"].diff().dt.days.dropna()
print("\n=== Distribusi selisih hari antar baris berturutan (df_joined) ===")
print(date_diff.value_counts().sort_index())
print(f"\nJumlah gap > 3 hari (indikasi libur panjang/hari trading hilang): {(date_diff > 3).sum()}")


=== Missing Values — df_ohlcv (Model A) ===
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

=== Missing Values — df_joined (Model B) ===
Date              0
Open              0
High              0
Low               0
Close             0
Volume            0
sentiment_mean    0
dtype: int64

=== Distribusi selisih hari antar baris berturutan (df_joined) ===
Date
1.0     1183
2.0       23
3.0      268
4.0       51
5.0        1
11.0       1
Name: count, dtype: int64

Jumlah gap > 3 hari (indikasi libur panjang/hari trading hilang): 53


### Statistik Deskriptif

In [5]:
print("=== Statistik Deskriptif — Model A (OHLCV) ===")
print(df_ohlcv[OHLCV_FEATURE_COLS].describe())

print("\n=== Statistik Deskriptif — Model B (OHLCV + sentiment_mean) ===")
print(df_joined[FEATURE_COLS].describe())


=== Statistik Deskriptif — Model A (OHLCV) ===
              Open         High          Low        Close        Volume
count  1552.000000  1552.000000  1552.000000  1552.000000  1.552000e+03
mean   3624.003736  3645.425775  3601.236320  3624.701946  4.187904e+09
std     715.396398   718.031272   713.571875   716.230030  1.035816e+09
min    2290.709961  2300.729980  2191.860107  2237.399902  0.000000e+00
25%    2901.302490  2909.629944  2885.019897  2896.734985  3.549760e+09
50%    3729.674927  3759.044922  3705.510010  3729.540039  3.966020e+09
75%    4257.555176  4286.672607  4221.787354  4262.775146  4.560940e+09
max    5130.990234  5149.669922  5127.180176  5137.080078  9.976520e+09

=== Statistik Deskriptif — Model B (OHLCV + sentiment_mean) ===
              Open         High          Low        Close        Volume  \
count  1528.000000  1528.000000  1528.000000  1528.000000  1.528000e+03   
mean   3628.507210  3650.035774  3605.621936  3629.216453  4.195081e+09   
std     716.195

### Deteksi Outlier (Metode IQR)

In [6]:
def detect_outliers_iqr(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Deteksi outlier per kolom pakai metode IQR (1.5x IQR di luar Q1/Q3)."""
    rows = []
    for col in cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_outliers = int(((df[col] < lower) | (df[col] > upper)).sum())
        rows.append({
            "kolom": col, "Q1": q1, "Q3": q3, "IQR": iqr,
            "batas_bawah": lower, "batas_atas": upper,
            "n_outlier": n_outliers, "pct_outlier": round(100 * n_outliers / len(df), 2),
        })
    return pd.DataFrame(rows)


outlier_report_b = detect_outliers_iqr(df_joined, FEATURE_COLS)
print("=== Laporan Outlier (IQR method) — Model B ===")
print(outlier_report_b.to_string(index=False))

fig, axes = plt.subplots(1, len(FEATURE_COLS), figsize=(4 * len(FEATURE_COLS), 5))
for ax, col in zip(axes, FEATURE_COLS):
    sns.boxplot(y=df_joined[col], ax=ax)
    ax.set_title(col)
fig.tight_layout()
fig.savefig(EDA_DIR / "eda07_boxplot_features.png", dpi=100)
plt.close(fig)
print(f"\nBoxplot disimpan: {EDA_DIR / 'eda07_boxplot_features.png'}")


=== Laporan Outlier (IQR method) — Model B ===
         kolom            Q1           Q3          IQR   batas_bawah   batas_atas  n_outlier  pct_outlier
          Open  2.904565e+03 4.268860e+03 1.364295e+03  8.581227e+02 6.315302e+03          0         0.00
          High  2.911000e+03 4.292238e+03 1.381238e+03  8.391435e+02 6.364094e+03          0         0.00
           Low  2.887015e+03 4.232470e+03 1.345455e+03  8.688326e+02 6.250652e+03          0         0.00
         Close  2.900495e+03 4.273595e+03 1.373100e+03  8.408452e+02 6.333245e+03          0         0.00
        Volume  3.561815e+09 4.563752e+09 1.001938e+09  2.058909e+09 6.066659e+09         97         6.35
sentiment_mean -2.328242e-01 9.935647e-02 3.321807e-01 -7.310952e-01 5.976274e-01         44         2.88

Boxplot disimpan: e:\forecasting_index_snp500\stock_prediction\notebooks\eda07_boxplot_features.png


### Distribusi & Skewness/Kurtosis

In [7]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, FEATURE_COLS):
    sns.histplot(df_joined[col], kde=True, ax=ax)
    ax.set_title(col)
fig.tight_layout()
fig.savefig(EDA_DIR / "eda07_distributions.png", dpi=100)
plt.close(fig)
print(f"Histogram distribusi disimpan: {EDA_DIR / 'eda07_distributions.png'}")

skew_kurt = pd.DataFrame({
    "skewness": [stats.skew(df_joined[c]) for c in FEATURE_COLS],
    "kurtosis": [stats.kurtosis(df_joined[c]) for c in FEATURE_COLS],
}, index=FEATURE_COLS)
print("\n=== Skewness & Kurtosis per Fitur ===")
print(skew_kurt)


Histogram distribusi disimpan: e:\forecasting_index_snp500\stock_prediction\notebooks\eda07_distributions.png

=== Skewness & Kurtosis per Fitur ===
                skewness  kurtosis
Open            0.036956 -1.402886
High            0.033308 -1.421112
Low             0.039799 -1.380938
Close           0.037440 -1.399933
Volume          1.634143  4.521310
sentiment_mean  0.032556  1.053722


### Time Series Trend (Cek Non-Stasioneritas Visual)

In [8]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
axes[0].plot(df_joined["Date"], df_joined["Close"])
axes[0].set_title("Close Price Sepanjang Waktu (mentah, level absolut)")
axes[1].plot(df_joined["Date"], df_joined["Close"].rolling(30).mean(), color="orange")
axes[1].set_title("Rolling Mean 30-hari — Close (cek tren jangka panjang)")
axes[2].plot(df_joined["Date"], df_joined["Volume"], color="green")
axes[2].set_title("Volume Sepanjang Waktu")
fig.tight_layout()
fig.savefig(EDA_DIR / "eda07_timeseries.png", dpi=100)
plt.close(fig)
print(f"Plot time series disimpan: {EDA_DIR / 'eda07_timeseries.png'}")
print(
    "\nCatatan interpretasi: kalau rolling mean menunjukkan tren naik/turun jangka panjang "
    "yang jelas (non-stasioner), model akan kesulitan belajar pola prediktif arah HARIAN dari "
    "level harga mentah — salah satu hipotesis kenapa Model A/B collapse ke prediksi konstan."
)


Plot time series disimpan: e:\forecasting_index_snp500\stock_prediction\notebooks\eda07_timeseries.png

Catatan interpretasi: kalau rolling mean menunjukkan tren naik/turun jangka panjang yang jelas (non-stasioner), model akan kesulitan belajar pola prediktif arah HARIAN dari level harga mentah — salah satu hipotesis kenapa Model A/B collapse ke prediksi konstan.


### Correlation Matrix Antar Fitur

In [9]:
corr_matrix = df_joined[FEATURE_COLS].corr()
print("=== Correlation Matrix (Pearson) — Fitur Model B ===")
print(corr_matrix.round(3))

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap — Fitur Model B")
fig.tight_layout()
fig.savefig(EDA_DIR / "eda07_correlation_heatmap.png", dpi=100)
plt.close(fig)
print(f"\nHeatmap korelasi disimpan: {EDA_DIR / 'eda07_correlation_heatmap.png'}")


=== Correlation Matrix (Pearson) — Fitur Model B ===
                 Open   High    Low  Close  Volume  sentiment_mean
Open            1.000  1.000  0.999  0.999   0.075           0.007
High            1.000  1.000  0.999  0.999   0.084           0.008
Low             0.999  0.999  1.000  0.999   0.060           0.017
Close           0.999  0.999  0.999  1.000   0.071           0.017
Volume          0.075  0.084  0.060  0.071   1.000          -0.098
sentiment_mean  0.007  0.008  0.017  0.017  -0.098           1.000

Heatmap korelasi disimpan: e:\forecasting_index_snp500\stock_prediction\notebooks\eda07_correlation_heatmap.png


### Korelasi Fitur vs Label (Paling Krusial)

Label dihitung SEMENTARA di sini (horizon-1 hari, sama seperti Bagian 1b di bawah)
HANYA untuk analisis ini — tidak menggantikan `add_label()` yang dipanggil ulang nanti.


In [10]:
df_joined_eda = df_joined.copy()
df_joined_eda["label"] = (df_joined_eda["Close"].shift(-1) > df_joined_eda["Close"]).astype("float32")
df_joined_eda = df_joined_eda.iloc[:-1]

feature_label_corr = df_joined_eda[FEATURE_COLS + ["label"]].corr()["label"].drop("label")
feature_label_corr = feature_label_corr.reindex(feature_label_corr.abs().sort_values(ascending=False).index)

print("=== Korelasi Fitur terhadap Label (arah harga besok) ===")
print(feature_label_corr)
print(
    "\nCatatan interpretasi: korelasi mendekati 0 (mis. |r| < 0.05) berarti fitur ini SECARA "
    "LINIER hampir tidak mengandung informasi untuk memprediksi arah harga besok. Kalau SEMUA "
    "fitur korelasinya mendekati nol, itu bukti kuat kenapa model (apapun arsitekturnya) "
    "collapse ke prediksi kelas mayoritas — tidak ada sinyal linier untuk dipelajari. Konsisten "
    "dengan hipotesis Efficient Market Hypothesis yang dibahas di notebook 03/05/06 project ini."
)


=== Korelasi Fitur terhadap Label (arah harga besok) ===
High             -0.037816
Close            -0.037764
Open             -0.036655
Low              -0.036120
sentiment_mean    0.034052
Volume           -0.021205
Name: label, dtype: float64

Catatan interpretasi: korelasi mendekati 0 (mis. |r| < 0.05) berarti fitur ini SECARA LINIER hampir tidak mengandung informasi untuk memprediksi arah harga besok. Kalau SEMUA fitur korelasinya mendekati nol, itu bukti kuat kenapa model (apapun arsitekturnya) collapse ke prediksi kelas mayoritas — tidak ada sinyal linier untuk dipelajari. Konsisten dengan hipotesis Efficient Market Hypothesis yang dibahas di notebook 03/05/06 project ini.


### Label Balance per Split (Cek Distribution Drift)

In [11]:
n = len(df_joined_eda)
n_train = int(n * 0.70)
n_val = int(n * 0.15)
split_tags = pd.Series(
    ["train"] * n_train + ["val"] * n_val + ["test"] * (n - n_train - n_val),
    index=df_joined_eda.index,
)
label_balance = df_joined_eda.groupby(split_tags)["label"].mean().reindex(["train", "val", "test"])
print("=== Proporsi Label 'Naik' (1) per Split (kronologis 70/15/15) ===")
print(label_balance)

fig, ax = plt.subplots(figsize=(6, 4))
label_balance.plot(kind="bar", ax=ax, color=["steelblue", "orange", "green"])
ax.axhline(0.5, color="red", linestyle="--", label="50% (seimbang)")
ax.set_ylabel("Proporsi Label = 1 (Naik)")
ax.set_title("Label Balance per Split")
ax.legend()
fig.tight_layout()
fig.savefig(EDA_DIR / "eda07_label_balance_per_split.png", dpi=100)
plt.close(fig)
print(f"\nChart label balance disimpan: {EDA_DIR / 'eda07_label_balance_per_split.png'}")
print(
    "\nCatatan interpretasi: kalau proporsi label berbeda jauh antar split (label drift), "
    "model yang 'collapse' ke prediksi kelas mayoritas TRAIN akan tampak buruk di VAL/TEST "
    "karena distribusi kelasnya bergeser — salah satu penyebab val_acc & test metrics di "
    "Bagian 4-5 sebelumnya."
)


=== Proporsi Label 'Naik' (1) per Split (kronologis 70/15/15) ===
train    0.552434
val      0.454148
test     0.565217
Name: label, dtype: float32

Chart label balance disimpan: e:\forecasting_index_snp500\stock_prediction\notebooks\eda07_label_balance_per_split.png

Catatan interpretasi: kalau proporsi label berbeda jauh antar split (label drift), model yang 'collapse' ke prediksi kelas mayoritas TRAIN akan tampak buruk di VAL/TEST karena distribusi kelasnya bergeser — salah satu penyebab val_acc & test metrics di Bagian 4-5 sebelumnya.


## Label & Split 70/15/15

Label horizon-1 hari: `label = 1 jika Close[t+1] > Close[t] else 0` (baris terakhir dibuang
karena label-nya NaN akibat shift). TANPA filter zona netral — beda dari notebook 03/06 yang
horizon-5-hari dengan threshold ±1%.

Split kronologis **berdasarkan persentase baris** (70% train / 15% val / 15% test), BUKAN
by-date cutoff tetap seperti `build_dataset()` produksi, dan BUKAN walk-forward berulang —
satu split tetap, sesuai keputusan yang dikonfirmasi.


In [12]:
def add_label(df: pd.DataFrame) -> pd.DataFrame:
    """Tambah kolom label horizon-1 hari, buang baris terakhir (label NaN)."""
    out = df.copy()
    out["label"] = (out["Close"].shift(-1) > out["Close"]).astype("float32")
    out = out.iloc[:-1].reset_index(drop=True)
    return out


def split_chronological(df: pd.DataFrame, train_frac: float = 0.70, val_frac: float = 0.15):
    """Split kronologis berdasarkan persentase baris (bukan by-date cutoff)."""
    n = len(df)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)

    df_train = df.iloc[:n_train].reset_index(drop=True)
    df_val = df.iloc[n_train : n_train + n_val].reset_index(drop=True)
    df_test = df.iloc[n_train + n_val :].reset_index(drop=True)
    return df_train, df_val, df_test


df_a = add_label(df_ohlcv)
df_b = add_label(df_joined)

df_a_train, df_a_val, df_a_test = split_chronological(df_a)
df_b_train, df_b_val, df_b_test = split_chronological(df_b)

for name, d_train, d_val, d_test in [("Model A", df_a_train, df_a_val, df_a_test),
                                       ("Model B", df_b_train, df_b_val, df_b_test)]:
    print(f"--- {name} ---")
    print(f"  train: {len(d_train)} baris | label mean = {d_train['label'].mean():.4f}")
    print(f"  val:   {len(d_val)} baris | label mean = {d_val['label'].mean():.4f}")
    print(f"  test:  {len(d_test)} baris | label mean = {d_test['label'].mean():.4f}")


--- Model A ---
  train: 1085 baris | label mean = 0.5548
  val:   232 baris | label mean = 0.4526
  test:  234 baris | label mean = 0.5641
--- Model B ---
  train: 1068 baris | label mean = 0.5524
  val:   229 baris | label mean = 0.4541
  test:  230 baris | label mean = 0.5652


## Bagian 1e — Cleaning Outlier (Capping IQR)

Berdasarkan temuan EDA (Bagian 1a): `Volume` dan `sentiment_mean` punya outlier signifikan
(6.35% dan 2.88% dari data), sedangkan kolom harga OHLC = 0 outlier (tidak perlu diapa-apakan).

**Metode: capping/winsorization** — nilai di luar batas IQR (`Q1-1.5×IQR`, `Q3+1.5×IQR`)
di-*clip* ke batas tersebut, **BUKAN dibuang barisnya** — supaya kontinuitas sliding-window
time-series tidak rusak (menghapus baris akan membuat gap yang mengacaukan windowing).

**Anti-leakage:** batas IQR dihitung **HANYA dari `df_*_train`** (bukan dari keseluruhan
dataset seperti eksplorasi di Bagian 1a), lalu batas yang SAMA diterapkan ke val/test —
konsisten dengan prinsip `MinMaxScaler` fit-train-only yang sudah dipakai di seluruh notebook
project ini.

Dilakukan SETELAH split (Bagian 1b), SEBELUM scaling (di bawah). Variabel
`df_a_train/val/test` dan `df_b_train/val/test` di-*update* (overwrite dengan versi yang
sudah di-cap) supaya langkah Scaling otomatis memakai data bersih tanpa perlu mengubah cell
lain.


In [13]:
def compute_iqr_bounds(df_train: pd.DataFrame, cols: list[str]) -> dict:
    """Hitung batas IQR (Q1-1.5*IQR, Q3+1.5*IQR) HANYA dari train (anti-leakage)."""
    bounds = {}
    for col in cols:
        q1, q3 = df_train[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_capping(df: pd.DataFrame, bounds: dict) -> pd.DataFrame:
    """Clip kolom sesuai batas IQR yang sudah dihitung (dari train)."""
    out = df.copy()
    for col, (lower, upper) in bounds.items():
        out[col] = out[col].clip(lower=lower, upper=upper)
    return out


In [14]:
# --- Model A: hanya Volume yang perlu di-cap (OHLC = 0 outlier, lihat Bagian 1a) ---
bounds_a = compute_iqr_bounds(df_a_train, ["Volume"])
print("Batas capping Model A (dihitung dari df_a_train):")
for col, (lo, hi) in bounds_a.items():
    print(f"  {col}: [{lo:.2f}, {hi:.2f}]")

for name, d in [("train", df_a_train), ("val", df_a_val), ("test", df_a_test)]:
    n_capped = ((d["Volume"] < bounds_a["Volume"][0]) | (d["Volume"] > bounds_a["Volume"][1])).sum()
    print(f"  Model A — jumlah nilai Volume yang akan di-cap di {name}: {n_capped}")

df_a_train = apply_capping(df_a_train, bounds_a)
df_a_val = apply_capping(df_a_val, bounds_a)
df_a_test = apply_capping(df_a_test, bounds_a)


Batas capping Model A (dihitung dari df_a_train):
  Volume: [1697485000.00, 6398405000.00]
  Model A — jumlah nilai Volume yang akan di-cap di train: 60
  Model A — jumlah nilai Volume yang akan di-cap di val: 9
  Model A — jumlah nilai Volume yang akan di-cap di test: 5


In [15]:
# --- Model B: Volume + sentiment_mean ---
bounds_b = compute_iqr_bounds(df_b_train, ["Volume", "sentiment_mean"])
print("Batas capping Model B (dihitung dari df_b_train):")
for col, (lo, hi) in bounds_b.items():
    print(f"  {col}: [{lo:.4f}, {hi:.4f}]")

for name, d in [("train", df_b_train), ("val", df_b_val), ("test", df_b_test)]:
    for col in ["Volume", "sentiment_mean"]:
        n_capped = ((d[col] < bounds_b[col][0]) | (d[col] > bounds_b[col][1])).sum()
        print(f"  Model B — jumlah nilai {col} yang akan di-cap di {name}: {n_capped}")

df_b_train = apply_capping(df_b_train, bounds_b)
df_b_val = apply_capping(df_b_val, bounds_b)
df_b_test = apply_capping(df_b_test, bounds_b)


Batas capping Model B (dihitung dari df_b_train):
  Volume: [1708866250.0000, 6395296250.0000]
  sentiment_mean: [-0.7280, 0.6666]
  Model B — jumlah nilai Volume yang akan di-cap di train: 61
  Model B — jumlah nilai sentiment_mean yang akan di-cap di train: 36
  Model B — jumlah nilai Volume yang akan di-cap di val: 10
  Model B — jumlah nilai sentiment_mean yang akan di-cap di val: 0
  Model B — jumlah nilai Volume yang akan di-cap di test: 5
  Model B — jumlah nilai sentiment_mean yang akan di-cap di test: 1


### Verifikasi Setelah Capping

Reuse `detect_outliers_iqr()` (didefinisikan di Bagian 1a) pada `df_b_train` yang sudah
di-cap — seharusnya `n_outlier = 0` untuk `Volume` dan `sentiment_mean` sekarang.


In [16]:
verify_report = detect_outliers_iqr(df_b_train, ["Volume", "sentiment_mean"])
print("=== Verifikasi Outlier Setelah Capping (df_b_train) ===")
print(verify_report.to_string(index=False))
assert (verify_report["n_outlier"] == 0).all(), "Masih ada outlier setelah capping — cek logic!"
print("\nOK — tidak ada outlier tersisa di df_b_train setelah capping.")


=== Verifikasi Outlier Setelah Capping (df_b_train) ===
         kolom            Q1           Q3          IQR   batas_bawah   batas_atas  n_outlier  pct_outlier
        Volume  3.466278e+09 4.637885e+09 1.171608e+09  1.708866e+09 6.395296e+09          0          0.0
sentiment_mean -2.050336e-01 1.436117e-01 3.486453e-01 -7.280016e-01 6.665797e-01          0          0.0

OK — tidak ada outlier tersisa di df_b_train setelah capping.


## Scaling

`MinMaxScaler` fit **HANYA di train** lalu transform val/test (anti-leakage — prinsip
metodologis universal, tetap dipertahankan meski bukan bagian eksplisit Batasan Masalah).
Model A dan Model B punya scaler terpisah karena jumlah fitur berbeda (5 vs 6 kolom).


In [17]:
from sklearn.preprocessing import MinMaxScaler


def fit_scale(df_train, df_val, df_test, feature_cols):
    scaler = MinMaxScaler()
    train_feat = scaler.fit_transform(df_train[feature_cols].values.astype(np.float64))
    val_feat = scaler.transform(df_val[feature_cols].values.astype(np.float64))
    test_feat = scaler.transform(df_test[feature_cols].values.astype(np.float64))
    return scaler, train_feat, val_feat, test_feat


scaler_a, feat_a_train, feat_a_val, feat_a_test = fit_scale(df_a_train, df_a_val, df_a_test, OHLCV_FEATURE_COLS)
scaler_b, feat_b_train, feat_b_val, feat_b_test = fit_scale(df_b_train, df_b_val, df_b_test, FEATURE_COLS)

print("Model A scaled shapes:", feat_a_train.shape, feat_a_val.shape, feat_a_test.shape)
print("Model B scaled shapes:", feat_b_train.shape, feat_b_val.shape, feat_b_test.shape)


Model A scaled shapes: (1085, 5) (232, 5) (234, 5)
Model B scaled shapes: (1068, 6) (229, 6) (230, 6)


## Windowing

Sliding window **fixed `WINDOW = 10`** (tidak ikut di-*search*, sesuai keputusan yang
dikonfirmasi). `X[i] = features[i : i+WINDOW]`, `y[i] = label` pada hari terakhir window.
Windowing dilakukan **per-split** (tidak melintasi batas train/val/test).


In [18]:
WINDOW = 10


def make_windows(features: np.ndarray, labels: np.ndarray, window: int):
    n = len(features)
    if n < window:
        return (
            np.empty((0, window, features.shape[1]), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
        )
    num_samples = n - window + 1
    X = np.stack([features[i : i + window] for i in range(num_samples)]).astype(np.float32)
    y = labels[window - 1 : window - 1 + num_samples].astype(np.float32)
    return X, y


X_a_train, y_a_train = make_windows(feat_a_train, df_a_train["label"].values, WINDOW)
X_a_val, y_a_val = make_windows(feat_a_val, df_a_val["label"].values, WINDOW)
X_a_test, y_a_test = make_windows(feat_a_test, df_a_test["label"].values, WINDOW)

X_b_train, y_b_train = make_windows(feat_b_train, df_b_train["label"].values, WINDOW)
X_b_val, y_b_val = make_windows(feat_b_val, df_b_val["label"].values, WINDOW)
X_b_test, y_b_test = make_windows(feat_b_test, df_b_test["label"].values, WINDOW)

print("Model A — X_train:", X_a_train.shape, "X_val:", X_a_val.shape, "X_test:", X_a_test.shape)
print("Model B — X_train:", X_b_train.shape, "X_val:", X_b_val.shape, "X_test:", X_b_test.shape)


Model A — X_train: (1076, 10, 5) X_val: (223, 10, 5) X_test: (225, 10, 5)
Model B — X_train: (1059, 10, 6) X_val: (220, 10, 6) X_test: (221, 10, 6)


## Bagian 2 — Arsitektur LSTM

Satu class `FlexibleLSTMClassifier` dipakai untuk Model A & Model B (`input_size` beda:
5 vs 6). Arsitektur sengaja fleksibel (hidden size, jumlah layer, dropout, aktivasi FC-head
bisa dikonfigurasi) karena akan di-*search* pada Bagian 3 — bukan hardcode satu konfigurasi
tetap seperti `LSTMDirectionClassifier`/`LSTMSoftmaxClassifier` produksi.

- Output LSTM: **last-timestep** (`lstm_out[:, -1, :]`), bukan mean-pooling.
- Output akhir: **2-class softmax** (`Linear(16, 2)`), gunakan `CrossEntropyLoss` saat
  training — TANPA softmax eksplisit di dalam `forward()` (`CrossEntropyLoss` menerima
  logit mentah). Prediksi kelas via `argmax`.

**Update (2026-07-14, atas keputusan user):** semula didesain 1-logit + `BCEWithLogitsLoss`
+ `pos_weight` (lihat log keputusan Bagian 3) — diganti ke 2-class softmax +
`CrossEntropyLoss` + class-weight per-kelas (inverse-frequency), supaya arsitektur
konsisten dengan rencana implementasi live-training di BE/FE (`ui-stock-prediction`,
tab "Model Training") yang memakai granularitas SSE per-epoch bergaya `LSTMSoftmaxClassifier`
produksi. Perubahan ini TIDAK mengubah hasil evaluasi (tetap HANYA Accuracy/Precision/
Recall/F1) maupun keputusan metodologis lain (Random Search, capping outlier, split
70/15/15) — murni pergantian formulasi loss/output layer.


In [19]:
ACTIVATION_MAP = {
    "relu": nn.ReLU,
    "tanh": nn.Tanh,
    "sigmoid": nn.Sigmoid,
}


class FlexibleLSTMClassifier(nn.Module):
    """LSTM classifier dgn arsitektur bisa dikonfigurasi untuk Random Search.

    Output LSTM: last-timestep. Head: Linear -> activation -> Dropout -> Linear(num_classes).
    forward() mengembalikan logit mentah (B, num_classes), TANPA softmax — gunakan
    CrossEntropyLoss saat training, argmax(dim=1) saat prediksi.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int = 32,
        num_layers: int = 1,
        dropout: float = 0.3,
        activation: str = "relu",
        num_classes: int = 2,
    ) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )
        act_cls = ACTIVATION_MAP[activation]
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 16),
            act_cls(),
            nn.Dropout(dropout),
            nn.Linear(16, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(x)          # (B, W, hidden)
        last_hidden = lstm_out[:, -1, :]    # (B, hidden) — last-timestep
        logits = self.head(last_hidden)     # (B, num_classes)
        return logits


## Bagian 3 — Hyperparameter Tuning: Random Search

Sesuai Batasan Masalah poin 4, search space mencakup: jumlah units per layer, jumlah layer,
dropout rate, jenis optimizer, fungsi aktivasi (FC-head), learning rate, jumlah epochs, dan
batch size. **20 iterasi** random sampling per model (bukan cartesian grid penuh), dijalankan
terpisah untuk Model A dan Model B (hyperparameter optimal boleh berbeda antar model).

Kriteria seleksi: ~~Val Accuracy~~ **Val F1-Score** (diubah 2026-07-14 atas keputusan user,
setelah ditemukan label drift antar split train/val/test lewat EDA Bagian 1a — val accuracy
rentan memilih config yang collapse ke kelas mayoritas kalau distribusi label val kebetulan
berbeda dari test). F1 tetap salah satu dari 4 metrik resmi yang diizinkan Batasan Masalah
BAB I, jadi perubahan ini **tidak menambah metrik baru di luar scope** — hanya mengganti
kriteria PEMILIHAN config, bukan metrik EVALUASI akhir (evaluasi akhir di Bagian 4 tetap
HANYA Accuracy/Precision/Recall/F1).

**Perbaikan tambahan (2026-07-14, aman terhadap scope — murni perbaikan prosedur training,
bukan perubahan fitur):**
- **Class weighting**: `pos_weight` di `BCEWithLogitsLoss` (dihitung dari rasio kelas TRAIN)
  supaya optimizer tidak mengambil jalan pintas collapse ke kelas mayoritas.
- **Fix bug seed**: sebelumnya `train_and_eval()` dipanggil dengan `seed=42` HARDCODE di
  setiap dari 20 iterasi Random Search (bug) — kini `seed` bervariasi per iterasi
  (`seed_dasar + i`) supaya inisialisasi bobot benar-benar berbeda antar konfigurasi, sesuai
  semangat "Random Search" yang sebenarnya.


In [20]:
SEARCH_SPACE = {
    "hidden_size": [16, 32, 64, 128],
    "num_layers": [1, 2],
    "dropout": (0.2, 0.5),          # uniform continuous
    "optimizer": ["adam", "rmsprop", "sgd"],
    "activation": ["relu", "tanh", "sigmoid"],
    "lr": (1e-4, 1e-2),             # log-uniform continuous
    "epochs": [30, 50, 80],
    "batch_size": [16, 32, 64],
}

N_ITER = 20


def sample_config(rng: np.random.Generator) -> dict:
    dropout_lo, dropout_hi = SEARCH_SPACE["dropout"]
    lr_lo, lr_hi = SEARCH_SPACE["lr"]
    log_lr = rng.uniform(np.log10(lr_lo), np.log10(lr_hi))
    return {
        "hidden_size": int(rng.choice(SEARCH_SPACE["hidden_size"])),
        "num_layers": int(rng.choice(SEARCH_SPACE["num_layers"])),
        "dropout": float(rng.uniform(dropout_lo, dropout_hi)),
        "optimizer": str(rng.choice(SEARCH_SPACE["optimizer"])),
        "activation": str(rng.choice(SEARCH_SPACE["activation"])),
        "lr": float(10 ** log_lr),
        "epochs": int(rng.choice(SEARCH_SPACE["epochs"])),
        "batch_size": int(rng.choice(SEARCH_SPACE["batch_size"])),
    }


def build_optimizer(name: str, params, lr: float):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr)
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr)
    raise ValueError(f"Optimizer tidak dikenal: {name}")


In [21]:
from sklearn.metrics import f1_score


def compute_class_weights(y_train: np.ndarray) -> torch.Tensor:
    """Inverse-frequency class weights (2 kelas), dihitung dari TRAIN saja."""
    counts = np.bincount(y_train.astype(np.int64), minlength=2)
    w = 1.0 / (counts + 1e-9)
    w = w / w.sum() * len(counts)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def train_and_eval(config: dict, input_size: int, X_train, y_train, X_val, y_val, seed: int = 42):
    """Latih satu model sesuai config, kembalikan (model, val_acc, val_f1, history).

    Perbaikan (2026-07-14):
    - Class weighting per-kelas (inverse-frequency, dihitung dari TRAIN saja) di
      `CrossEntropyLoss` — supaya optimizer tidak mengambil jalan pintas collapse ke
      kelas mayoritas.
    - Mengembalikan val_f1 sekaligus val_acc — val_f1 dipakai sbg kriteria seleksi
      di run_random_search (lihat catatan Bagian 3), val_acc tetap ditampilkan.

    Update (2026-07-14): loss diganti BCEWithLogitsLoss+pos_weight -> CrossEntropyLoss+
    class-weight per-kelas, mengikuti perubahan arsitektur FlexibleLSTMClassifier ke
    2-class softmax (lihat markdown Bagian 2). Target di-cast ke int64 (LongTensor) —
    syarat CrossEntropyLoss.
    """
    set_seed(seed)

    model = FlexibleLSTMClassifier(
        input_size=input_size,
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
        activation=config["activation"],
        num_classes=2,
    ).to(DEVICE)

    optimizer = build_optimizer(config["optimizer"], model.parameters(), config["lr"])

    class_weights = compute_class_weights(y_train)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    y_train_long = y_train.astype(np.int64)
    y_val_long = y_val.astype(np.int64)

    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train_long))
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)

    X_val_t = torch.from_numpy(X_val).to(DEVICE)
    y_val_t = torch.from_numpy(y_val_long).to(DEVICE)

    history = {"train_loss": []}

    model.train()
    for epoch in range(config["epochs"]):
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)
        history["train_loss"].append(epoch_loss / len(train_ds))

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t)
        val_pred = val_logits.argmax(dim=1)
        val_acc = (val_pred == y_val_t).float().mean().item()
        val_f1 = f1_score(y_val_t.cpu().numpy(), val_pred.cpu().numpy(), zero_division=0)

    return model, val_acc, val_f1, history


def run_random_search(model_name: str, input_size: int, X_train, y_train, X_val, y_val, n_iter: int = N_ITER, seed: int = 42):
    """Random Search — kriteria seleksi: Val F1 (lihat catatan Bagian 3).

    Perbaikan (2026-07-14): seed training kini bervariasi per iterasi (`seed + i`),
    bukan hardcode 42 — supaya inisialisasi bobot benar-benar berbeda antar konfigurasi.
    """
    rng = np.random.default_rng(seed)
    results = []
    best = {"val_f1": -1.0, "val_acc": -1.0, "config": None, "model": None}

    for i in range(n_iter):
        config = sample_config(rng)
        model, val_acc, val_f1, _ = train_and_eval(
            config, input_size, X_train, y_train, X_val, y_val, seed=seed + i
        )
        results.append({**config, "val_acc": val_acc, "val_f1": val_f1})
        logger.info(
            "[%s] iter %d/%d — val_f1=%.4f val_acc=%.4f — config=%s",
            model_name, i + 1, n_iter, val_f1, val_acc, config,
        )

        if val_f1 > best["val_f1"]:
            best = {"val_f1": val_f1, "val_acc": val_acc, "config": config, "model": model}

    results_df = pd.DataFrame(results).sort_values("val_f1", ascending=False).reset_index(drop=True)
    return best, results_df


In [22]:
best_a, results_a = run_random_search(
    "Model A (OHLCV)", input_size=len(OHLCV_FEATURE_COLS),
    X_train=X_a_train, y_train=y_a_train, X_val=X_a_val, y_val=y_a_val,
    n_iter=N_ITER, seed=42,
)
print("=== Model A — Top 5 hasil Random Search (sorted by Val F1) ===")
print(results_a.head(5))
print("\nBest config Model A:", best_a["config"], "| val_f1 =", best_a["val_f1"], "| val_acc =", best_a["val_acc"])


2026-07-23 18:05:32,439 [INFO] notebook07: [Model A (OHLCV)] iter 1/20 — val_f1=0.6192 val_acc=0.4484 — config={'hidden_size': 64, 'num_layers': 1, 'dropout': 0.45757937597341475, 'optimizer': 'adam', 'activation': 'sigmoid', 'lr': 0.00353111691382141, 'epochs': 30, 'batch_size': 16}
2026-07-23 18:05:43,185 [INFO] notebook07: [Model A (OHLCV)] iter 2/20 — val_f1=0.6192 val_acc=0.4484 — config={'hidden_size': 64, 'num_layers': 2, 'dropout': 0.43581929158308613, 'optimizer': 'rmsprop', 'activation': 'relu', 'lr': 0.008938089586343595, 'epochs': 80, 'batch_size': 32}
2026-07-23 18:05:49,733 [INFO] notebook07: [Model A (OHLCV)] iter 3/20 — val_f1=0.6192 val_acc=0.4484 — config={'hidden_size': 16, 'num_layers': 2, 'dropout': 0.3931595360241994, 'optimizer': 'rmsprop', 'activation': 'sigmoid', 'lr': 0.0005515641726960964, 'epochs': 50, 'batch_size': 32}
2026-07-23 18:05:52,946 [INFO] notebook07: [Model A (OHLCV)] iter 4/20 — val_f1=0.6192 val_acc=0.4484 — config={'hidden_size': 16, 'num_laye

=== Model A — Top 5 hasil Random Search (sorted by Val F1) ===
   hidden_size  num_layers   dropout optimizer activation        lr  epochs  \
0           64           1  0.457579      adam    sigmoid  0.003531      30   
1           64           2  0.435819   rmsprop       relu  0.008938      80   
2           16           2  0.393160   rmsprop    sigmoid  0.000552      50   
3           16           2  0.219145       sgd    sigmoid  0.000285      30   
4          128           1  0.436077       sgd       tanh  0.000251      50   

   batch_size   val_acc    val_f1  
0          16  0.448431  0.619195  
1          32  0.448431  0.619195  
2          32  0.448431  0.619195  
3          32  0.448431  0.619195  
4          64  0.448431  0.619195  

Best config Model A: {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.45757937597341475, 'optimizer': 'adam', 'activation': 'sigmoid', 'lr': 0.00353111691382141, 'epochs': 30, 'batch_size': 16} | val_f1 = 0.6191950464396285 | val_acc = 0.448430

In [23]:
best_b, results_b = run_random_search(
    "Model B (FinBERT-LSTM)", input_size=len(FEATURE_COLS),
    X_train=X_b_train, y_train=y_b_train, X_val=X_b_val, y_val=y_b_val,
    n_iter=N_ITER, seed=43,
)
print("=== Model B — Top 5 hasil Random Search (sorted by Val F1) ===")
print(results_b.head(5))
print("\nBest config Model B:", best_b["config"], "| val_f1 =", best_b["val_f1"], "| val_acc =", best_b["val_acc"])


2026-07-23 18:07:45,217 [INFO] notebook07: [Model B (FinBERT-LSTM)] iter 1/20 — val_f1=0.5865 val_acc=0.5000 — config={'hidden_size': 32, 'num_layers': 1, 'dropout': 0.20600887606226506, 'optimizer': 'adam', 'activation': 'sigmoid', 'lr': 0.0020165013885763234, 'epochs': 50, 'batch_size': 32}
2026-07-23 18:07:51,216 [INFO] notebook07: [Model B (FinBERT-LSTM)] iter 2/20 — val_f1=0.0741 val_acc=0.5455 — config={'hidden_size': 128, 'num_layers': 2, 'dropout': 0.27910765924351244, 'optimizer': 'adam', 'activation': 'tanh', 'lr': 0.00028145596704174725, 'epochs': 50, 'batch_size': 32}
2026-07-23 18:07:59,691 [INFO] notebook07: [Model B (FinBERT-LSTM)] iter 3/20 — val_f1=0.6207 val_acc=0.4500 — config={'hidden_size': 32, 'num_layers': 2, 'dropout': 0.28358990765568304, 'optimizer': 'sgd', 'activation': 'relu', 'lr': 0.008140089096539387, 'epochs': 80, 'batch_size': 32}
2026-07-23 18:08:05,913 [INFO] notebook07: [Model B (FinBERT-LSTM)] iter 4/20 — val_f1=0.0000 val_acc=0.5500 — config={'hidd

=== Model B — Top 5 hasil Random Search (sorted by Val F1) ===
   hidden_size  num_layers   dropout optimizer activation        lr  epochs  \
0           32           2  0.283590       sgd       relu  0.008140      80   
1          128           2  0.463645       sgd    sigmoid  0.001633      80   
2           32           1  0.372089       sgd    sigmoid  0.004855      50   
3          128           2  0.377952       sgd    sigmoid  0.000935      80   
4          128           2  0.409638       sgd    sigmoid  0.001012      50   

   batch_size  val_acc   val_f1  
0          32     0.45  0.62069  
1          32     0.45  0.62069  
2          64     0.45  0.62069  
3          64     0.45  0.62069  
4          16     0.45  0.62069  

Best config Model B: {'hidden_size': 32, 'num_layers': 2, 'dropout': 0.28358990765568304, 'optimizer': 'sgd', 'activation': 'relu', 'lr': 0.008140089096539387, 'epochs': 80, 'batch_size': 32} | val_f1 = 0.6206896551724138 | val_acc = 0.44999998807907104


## Bagian 4 — Evaluasi Test-Set (HANYA 4 Metrik)

Model final dipakai adalah model dengan val_accuracy tertinggi selama Random Search
(bukan dilatih ulang dari nol) — konsisten dengan config yang menghasilkan performa
val terbaik, menghindari mismatch acak antara pemilihan config dan model final.

Evaluasi test-set **HANYA**: Accuracy, Precision, Recall, F1-Score (`sklearn.metrics`).
**TIDAK** menghitung AUC-ROC, class-collapse diagnosis, atau metrik lain di luar itu —
supaya notebook match 1:1 dengan Tujuan Khusus BAB I. Confusion matrix mentah (tn/fp/fn/tp)
tetap ditampilkan sebagai pendukung angka precision/recall (lazim di BAB IV skripsi), BUKAN
sebagai metrik tambahan.


In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


def evaluate_model(model: nn.Module, X_test: np.ndarray, y_test: np.ndarray) -> dict:
    """Evaluasi test-set. Update (2026-07-14): prediksi via argmax (2-class softmax),
    bukan sigmoid>=0.5 (1-logit) — mengikuti perubahan arsitektur ke CrossEntropy."""
    model.eval()
    X_test_t = torch.from_numpy(X_test).to(DEVICE)
    with torch.no_grad():
        logits = model(X_test_t)
        y_pred = logits.argmax(dim=1).cpu().numpy()

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


metrics_a = evaluate_model(best_a["model"], X_a_test, y_a_test)
print("=== Model A (LSTM-OHLCV baseline) — Test Set ===")
for k, v in metrics_a.items():
    print(f"  {k}: {v}")


=== Model A (LSTM-OHLCV baseline) — Test Set ===
  accuracy: 0.56
  precision: 0.56
  recall: 1.0
  f1: 0.717948717948718
  confusion_matrix: {'tn': 0, 'fp': 99, 'fn': 0, 'tp': 126}


In [25]:
metrics_b = evaluate_model(best_b["model"], X_b_test, y_b_test)
print("=== Model B (FinBERT-LSTM) — Test Set ===")
for k, v in metrics_b.items():
    print(f"  {k}: {v}")


=== Model B (FinBERT-LSTM) — Test Set ===
  accuracy: 0.5610859728506787
  precision: 0.5610859728506787
  recall: 1.0
  f1: 0.7188405797101449
  confusion_matrix: {'tn': 0, 'fp': 97, 'fn': 0, 'tp': 124}


## Bagian 5 — Perbandingan Model A vs Model B (Menjawab Hipotesa #4 BAB I)

Rumusan Masalah #4 / Hipotesa utama: apakah FinBERT-LSTM (OHLCV + sentimen) mengungguli
LSTM murni (OHLCV saja) pada keempat metrik? Dilaporkan apa adanya — bukan berarti Model B
harus menang.


In [26]:
comparison_df = pd.DataFrame({
    "Metrik": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Model A (LSTM-OHLCV)": [metrics_a["accuracy"], metrics_a["precision"], metrics_a["recall"], metrics_a["f1"]],
    "Model B (FinBERT-LSTM)": [metrics_b["accuracy"], metrics_b["precision"], metrics_b["recall"], metrics_b["f1"]],
})
comparison_df["Selisih (B - A)"] = comparison_df["Model B (FinBERT-LSTM)"] - comparison_df["Model A (LSTM-OHLCV)"]
print(comparison_df.to_string(index=False))

n_metrics_b_wins = (comparison_df["Selisih (B - A)"] > 0).sum()
print(f"\nModel B unggul pada {n_metrics_b_wins}/4 metrik dibanding Model A.")
print("Confusion matrix Model A:", metrics_a["confusion_matrix"])
print("Confusion matrix Model B:", metrics_b["confusion_matrix"])
print(
    "\nCatatan: hasil dilaporkan apa adanya sesuai prinsip anti p-hacking — "
    "kesimpulan akhir (apakah sentimen FinBERT membantu prediksi arah harga) "
    "harus dinarasikan di BAB IV/V berdasarkan angka di atas, bukan diasumsikan sebelumnya."
)


   Metrik  Model A (LSTM-OHLCV)  Model B (FinBERT-LSTM)  Selisih (B - A)
 Accuracy              0.560000                0.561086         0.001086
Precision              0.560000                0.561086         0.001086
   Recall              1.000000                1.000000         0.000000
 F1-Score              0.717949                0.718841         0.000892

Model B unggul pada 3/4 metrik dibanding Model A.
Confusion matrix Model A: {'tn': 0, 'fp': 99, 'fn': 0, 'tp': 126}
Confusion matrix Model B: {'tn': 0, 'fp': 97, 'fn': 0, 'tp': 124}

Catatan: hasil dilaporkan apa adanya sesuai prinsip anti p-hacking — kesimpulan akhir (apakah sentimen FinBERT membantu prediksi arah harga) harus dinarasikan di BAB IV/V berdasarkan angka di atas, bukan diasumsikan sebelumnya.


## Bagian 6 — Simpan Artefak

Nama file terisolasi (`*_skripsi.*`) — TIDAK menimpa artefak produksi (`direction_classifier.pt`)
maupun artefak notebook 03/05/06 manapun. Disimpan di folder yang sama
(`data/artifacts/lstm_weights/`) mengikuti konvensi penamaan project ini.


In [27]:
import json
from datetime import datetime, timezone

import joblib

ARTIFACT_DIR = PROJECT_ROOT / "data" / "artifacts" / "lstm_weights"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_artifact(name: str, model: nn.Module, config: dict, scaler, metrics: dict, feature_cols: list[str]):
    model_path = ARTIFACT_DIR / f"{name}.pt"
    scaler_path = ARTIFACT_DIR / f"{name}_scaler.joblib"
    metadata_path = ARTIFACT_DIR / f"{name}_metadata.json"

    torch.save({
        "model_state_dict": model.state_dict(),
        "config": config,
        "feature_cols": feature_cols,
        "window": WINDOW,
    }, model_path)

    joblib.dump(scaler, scaler_path)

    metadata = {
        "name": name,
        "feature_cols": feature_cols,
        "window": WINDOW,
        "config": config,
        "metrics": metrics,
        "trained_at": datetime.now(timezone.utc).isoformat(),
    }
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, default=str)

    print(f"Saved: {model_path.name}, {scaler_path.name}, {metadata_path.name}")


save_artifact("lstm_baseline_skripsi", best_a["model"], best_a["config"], scaler_a, metrics_a, OHLCV_FEATURE_COLS)
save_artifact("finbert_lstm_skripsi", best_b["model"], best_b["config"], scaler_b, metrics_b, FEATURE_COLS)


Saved: lstm_baseline_skripsi.pt, lstm_baseline_skripsi_scaler.joblib, lstm_baseline_skripsi_metadata.json
Saved: finbert_lstm_skripsi.pt, finbert_lstm_skripsi_scaler.joblib, finbert_lstm_skripsi_metadata.json
